# 24. Scaleworm Population Time Series

<span style="font-family: 'Courier New', monospace;">

*AI-generated draft (Claude, Anthropic) — for review. All parameters and figures are derived from version-controlled scripts and data.*

Counts scaleworms across a month of CAMHD videos. For each video we scan a **window of frames** (185–325 s, every 5 s) and take the **maximum detection count** across that window. This handles the fact that different videos zoom in at slightly different timestamps, and that different time slots show slightly different camera angles. Videos where the camera never zoomed in (max = 0) are skipped in the plot.

**Workflow:**
1. Run **Cell 1** — loads model, defines configuration
2. Run **Cell 2** — discovers all `.mp4` files for the target month
3. Run **Cell 3** — processes each video (resumable: already-done videos are skipped)
4. Run **Cell 4** — loads saved results and plots the time series

To switch models, change `MODEL_PATH` in Cell 1 and re-run Cells 3–4.

</span>

## Known limitations and sources of inconsistency

<span style="font-family: 'Courier New', monospace;">

*AI-generated draft (Claude, Anthropic) — for review. All parameters and figures are derived from version-controlled scripts and data.*

The following inconsistencies were identified during development (2026-06-28) and have not yet been fully validated. Keep them in mind when interpreting results.

**1. Mixed camera styles in training data**
The v1 model was trained on frames from at least two distinct camera angles/zoom levels: a front-on close-up style (T061500-style videos) and a wider horizontal style (T091500-style videos). The proportion of each style in the training set is unknown, so the model may be systematically stronger at detecting worms in one camera style than the other.

**2. Zero-detection videos are dropped from the plot**
Videos where `max_count = 0` are excluded on the assumption the camera was not zoomed in on the vent. Some of these may instead be cases where the camera *was* zoomed in but the model failed to detect worms due to an unfamiliar angle, lighting, or turbidity. Check a sample of zero-detection videos before drawing conclusions about absence.

**3. Scan window is an empirical estimate**
The 185–325 s window was chosen based on one reference video (T091500, t=190 s). Zoom timing varies across videos — some zooms may start late or end early and fall partly or wholly outside the window, resulting in an undercount.

**What to do if results look suspicious:**
- Clusters of zeros or anomalously low counts at certain times of day → check raw frames for that video
- Systematic low counts for one camera style → check training data composition
- Counts that drop off at the start or end of the window → widen `WIN_START`/`WIN_END`

</span>

In [ ]:
import subprocess, json, sys, time
import tempfile
from pathlib import Path
from datetime import datetime

from ultralytics import YOLO

# ── Configuration — edit these as models improve ───────────────────
MODEL_PATH = Path(
    "runs/detect/verification_session/runs/scaleworm_retrained_v1/weights/best.pt"
)
VIDEO_DIR  = Path("/home/jovyan/ooi/san_data/RS03ASHS-PN03B-06-CAMHDA301/2024/10")
RESULTS_DIR = Path("./timeseries_results")

# Scan window: camera typically zooms into Mushroom vent somewhere in here
WIN_START  = 185   # seconds
WIN_END    = 325   # seconds
WIN_STEP   = 5     # seconds between sampled frames

CONF_THRESHOLD = 0.25  # YOLO confidence cutoff
# ──────────────────────────────────────────────────────────────────

RESULTS_DIR.mkdir(exist_ok=True)

model = YOLO(str(MODEL_PATH))
n_frames = len(range(WIN_START, WIN_END + 1, WIN_STEP))
print(f"Model   : {MODEL_PATH}")
print(f"Window  : {WIN_START}s – {WIN_END}s every {WIN_STEP}s  ({n_frames} frames/video)")
print(f"Results : {RESULTS_DIR.resolve()}")

In [ ]:
videos = sorted(VIDEO_DIR.rglob("*.mp4"))
already_done = len(list(RESULTS_DIR.glob("*.json")))
print(f"Found {len(videos)} videos in {VIDEO_DIR}")
print(f"Already processed: {already_done}  (will be skipped)")
print(f"Remaining        : {len(videos) - already_done}")
print()
for v in videos[:5]:
    print(f"  {v.name}")
print("  ...")

In [ ]:
def _parse_dt(video_path):
    """Extract datetime from CAMHDA301-YYYYMMDDTHHMMSS.mp4 filename."""
    dt_str = video_path.stem.split("-")[1]  # '20241001T091500'
    return datetime.strptime(dt_str, "%Y%m%dT%H%M%S")


def process_video(video_path, model, win_start, win_end, win_step, conf_thr):
    result_file = RESULTS_DIR / f"{video_path.stem}.json"
    if result_file.exists():
        return json.loads(result_file.read_text())  # resume: skip already done

    dt = _parse_dt(video_path)

    with tempfile.TemporaryDirectory() as tmp:
        tmp = Path(tmp)

        # Single ffmpeg pass: extract one frame every win_step seconds across the window
        ret = subprocess.run(
            [
                "ffmpeg", "-y",
                "-ss", str(win_start),
                "-to", str(win_end + win_step),
                "-i", str(video_path),
                "-vf", f"fps=1/{win_step}",
                "-q:v", "2",
                str(tmp / "frame_%04d.png"),
            ],
            capture_output=True,
            timeout=180,
        )

        frames = sorted(tmp.glob("frame_*.png"))

        if not frames:
            result = {
                "video": video_path.name,
                "dt": dt.isoformat(),
                "max_count": 0,
                "best_t": None,
                "frame_counts": [],
            }
            result_file.write_text(json.dumps(result))
            return result

        # Map extracted frames to approximate timestamps
        frame_ts = [(frames[i], win_start + i * win_step) for i in range(len(frames))]

        frame_counts = []
        for img_path, t in frame_ts:
            preds = model.predict(str(img_path), conf=conf_thr, verbose=False)
            # Count only scale_worm class (ignore not_worm / other classes)
            n = sum(
                1
                for box in preds[0].boxes
                if model.names[int(box.cls)] == "scale_worm"
            )
            frame_counts.append({"t": t, "count": n})

        max_count = max(fc["count"] for fc in frame_counts)
        best_t = next(
            (fc["t"] for fc in frame_counts if fc["count"] == max_count), None
        )

        result = {
            "video": video_path.name,
            "dt": dt.isoformat(),
            "max_count": max_count,
            "best_t": best_t,
            "frame_counts": frame_counts,
        }
        result_file.write_text(json.dumps(result))
        return result


# ── Run ────────────────────────────────────────────────────────────
all_results = []
t0 = time.time()

for i, vp in enumerate(videos):
    r = process_video(vp, model, WIN_START, WIN_END, WIN_STEP, CONF_THRESHOLD)
    all_results.append(r)
    elapsed = time.time() - t0
    done = i + 1
    remaining = len(videos) - done
    eta_min = (elapsed / done * remaining) / 60 if done > 0 else 0
    status = "(cached)" if (RESULTS_DIR / f"{vp.stem}.json").stat().st_size < 100 else ""
    print(
        f"[{done:3d}/{len(videos)}] {vp.name[:42]}  "
        f"max={r['max_count']:2d} worms @ t={r['best_t']}s  "
        f"ETA {eta_min:.1f} min  {status}"
    )

print(f"\nFinished {len(all_results)} videos in {(time.time()-t0)/60:.1f} min.")

In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "plotly"], check=True)
import plotly.graph_objects as go

# Load all saved results
saved = sorted(RESULTS_DIR.glob("*.json"))
all_results = [json.loads(f.read_text()) for f in saved]
all_results.sort(key=lambda r: r["dt"])

has_worms = [r for r in all_results if r["max_count"] > 0]
no_worms  = [r for r in all_results if r["max_count"] == 0]
print(f"Videos with ≥1 detection : {len(has_worms)}")
print(f"Videos with 0 detections : {len(no_worms)}  (camera likely not zoomed in — omitted from plot)")

dts    = [r["dt"] for r in has_worms]
counts = [r["max_count"] for r in has_worms]
hover  = [
    f"{r['video']}<br>Max detections: {r['max_count']}<br>Best frame: t={r['best_t']}s"
    for r in has_worms
]

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=dts,
    y=counts,
    mode="lines+markers",
    marker=dict(color="#0072B2", size=6),
    line=dict(color="#0072B2", width=1.5),
    hovertext=hover,
    hoverinfo="text+x",
    name="Max worms in scan window",
))

model_label = Path(MODEL_PATH).parts[-3] if len(Path(MODEL_PATH).parts) >= 3 else MODEL_PATH.stem

fig.update_layout(
    title=(
        f"Scaleworm count — October 2024<br>"
        f"<sub>Model: {model_label} | "
        f"Window: {WIN_START}–{WIN_END}s every {WIN_STEP}s | "
        f"Conf ≥ {CONF_THRESHOLD} | "
        f"{len(no_worms)} zero-detection videos omitted</sub>"
    ),
    xaxis_title="Video date/time (UTC)",
    yaxis_title="Scaleworm count (max in window)",
    template="plotly_white",
    height=520,
    hovermode="closest",
)
fig.show()